In [1]:
# dependencies

from Bio import AlignIO
from Bio import SeqIO
import re
import pandas as pd

In [2]:
# Converting BioCatNet fasta file to dataframe that includes the sequence and the family name

def biocatnet_fasta_to_df(input_fasta):
    data = {'accession': [], 'SFID': [], 'GI': [], 'taxonID': [], 'sequence': []}
    with open(input_fasta) as fp:
        for record in SeqIO.parse(fp, 'fasta'):
            accession = record.id
            sfid = re.findall(r"sfid\|(\d+)", record.id)
            gi = re.findall(r"gi\|(\d+)", record.id)
            taxonID = re.findall(r"taxonID\|(\d+)", record.id)
            sequence = str(record.seq)

            #Attach to dict.
            data['accession'].append(accession)
            data['SFID'].append(", ".join(sfid))
            data['GI'].append(", ".join(gi))
            data['taxonID'].append(", ".join(taxonID))
            data['sequence'].append(sequence)
    df = pd.DataFrame(data)
    return df

In [3]:
# Convert FASTA file into df of the results

df_results = biocatnet_fasta_to_df('/home/alecia/chapter3-repo/output/hmm_search_2/biocatnet/5/ihelixhits_e_0p00001_z__T_.fasta')
df_results

,accession,SFID,GI,taxonID,sequence
0,sid|107075|pid|82008|hfid|211|sfid|6|gb|ERL945...,6,546684943,77166,ITEEDI-MAQCFVFFFA------GFETSST--TMTFALLELAQH-
1,sid|107075|pid|82008|hfid|211|sfid|6|gb|ERL945...,6,546684943,77166,ISESDI-IAQCFVFFIA------GFETSSS--TMTFTMLELAQH-
2,sid|107075|pid|82008|hfid|211|sfid|6|gb|ERL945...,6,546684943,77166,ISESDI-IAQCFVFFIA------GFETSSS--TMTFTMLELAQH-
3,sid|107075|pid|82008|hfid|211|sfid|6|gb|ERL945...,6,546684943,77166,LSENDI-IGQCFVFFIA------GFETSSS--TMTFTMLELAQN-
4,sid|85794|pid|63405|hfid|120|sfid|4|gi|3544699...,4,"354469982, 537224038",10029,LSDKDL-RAEVDTFMFE------GHDTTAS--GISWIFYALATHP
...,...,...,...,...,...
27819,sid|8892|pid|6197|hfid|17|sfid|2|gi|488589356|...,2,"488589356, 488589360",9361,-----L-ILTGLSLFFA------GTETTST--TLRYGFLLMLKYP
27820,sid|4148|pid|2700|hfid|1066|sfid|150|gi|515044...,150,515044448,1829,----DL-VRLATFLFGA------GQDTTAR--LITAAFRLIAENP
27821,sid|79738|pid|58328|hfid|1592|sfid|706|gb|EYU2...,706,604301429,4155,----QI-KAILVDIVAG------GSDTTAT--MVEWVMTELLHNP
27822,sid|21392|pid|16201|hfid|653|sfid|82|gi|568865...,82,568865636,2711,-----I-KAICVTLIVA------AADTSVV--TLTWAIALLVNN-


In [4]:
# Open database and put into dataframe

df_database = biocatnet_fasta_to_df('/home/alecia/chapter3-repo/input/databases/biocatnet_p450.fasta')
df_database

,accession,SFID,GI,taxonID,sequence
0,sid|90155|pid|67114|hfid|29|sfid|2|gi|55729550...,2,557295506,38654,MDAGTTGILILLLLIFVLCYLVLEINRKRAQLPAGPAPWPVLGNLW...
1,sid|90154|pid|67113|hfid|29|sfid|2|gi|60264309...,2,602643091,176946,GPTPWPFLGNLLQRDVLPVDRLYKKLTAKYGPIFTVWIGSKPMVAL...
2,sid|90156|pid|67115|hfid|29|sfid|2|gi|60267100...,2,602671004,176946,MELTWAGALLLLCIFFTLFSSFQIYKKKGQLPPGPTPWPFLGNLLQ...
3,sid|90236|pid|67189|hfid|29|sfid|2|gb|ETE74148...,2,565323585,8665,MELTWTGVLLLLCVLFILFSSFQMYKKKGQLPPGPTPWPILGNLLQ...
4,sid|101660|pid|77115|hfid|29|sfid|2|gi|6026704...,2,602670425,176946,MELTWAGALLLLSILITLFSSFQMYKKKGQLPPGPTPWPFLGNLLQ...
...,...,...,...,...,...
52669,sid|14140|pid|10364|hfid|1674|sfid|5006|gi|118...,5006,"118351107, 164519821",5911,MSIIQLFIGFIIGTIIYQTVIKTLYLYYVYKRRYGKDIIVLFYPVI...
52670,sid|14148|pid|10372|hfid|1677|sfid|5009|gb|ABY...,5009,"164519833, 118380288",5911,MITTILIALTFIGIALLLFKAFIQPLYRISFYTKQGLKQKFFVPFL...
52671,sid|14504|pid|10663|hfid|1684|sfid|5060|gi|667...,5060,667644755,655819,MSSHKPPEAVRRETSLQGQSMTESHCMLALSERLHDPLPGCVTASI...
52672,sid|85661|pid|63286|hfid|1696|sfid|5280|gi|667...,5280,667652271,655819,MFMLWIAAAVFGLIIIGRPLIVYLQDVKGFRKYPVQNFLSGISPLA...


In [5]:
# Merge results with database to see which sequences are missing from results
## Accession cannot be used to merge on due to the presence of 'sub seq' 

merged_df = (pd.merge(df_database, df_results, how='left', on=['GI', 'SFID', 'taxonID'])
             .fillna(0))
merged_df

,accession_x,SFID,GI,taxonID,sequence_x,accession_y,sequence_y
0,sid|90155|pid|67114|hfid|29|sfid|2|gi|55729550...,2,557295506,38654,MDAGTTGILILLLLIFVLCYLVLEINRKRAQLPAGPAPWPVLGNLW...,0,0
1,sid|90154|pid|67113|hfid|29|sfid|2|gi|60264309...,2,602643091,176946,GPTPWPFLGNLLQRDVLPVDRLYKKLTAKYGPIFTVWIGSKPMVAL...,0,0
2,sid|90156|pid|67115|hfid|29|sfid|2|gi|60267100...,2,602671004,176946,MELTWAGALLLLCIFFTLFSSFQIYKKKGQLPPGPTPWPFLGNLLQ...,0,0
3,sid|90236|pid|67189|hfid|29|sfid|2|gb|ETE74148...,2,565323585,8665,MELTWTGVLLLLCVLFILFSSFQMYKKKGQLPPGPTPWPILGNLLQ...,0,0
4,sid|101660|pid|77115|hfid|29|sfid|2|gi|6026704...,2,602670425,176946,MELTWAGALLLLSILITLFSSFQMYKKKGQLPPGPTPWPFLGNLLQ...,0,0
...,...,...,...,...,...,...,...
52746,sid|14140|pid|10364|hfid|1674|sfid|5006|gi|118...,5006,"118351107, 164519821",5911,MSIIQLFIGFIIGTIIYQTVIKTLYLYYVYKRRYGKDIIVLFYPVI...,sid|14140|pid|10364|hfid|1674|sfid|5006|gi|118...,LTRKEI-VNQYITVIFA------GTDTTSH--LIGNILFELSRNP
52747,sid|14148|pid|10372|hfid|1677|sfid|5009|gb|ABY...,5009,"164519833, 118380288",5911,MITTILIALTFIGIALLLFKAFIQPLYRISFYTKQGLKQKFFVPFL...,0,0
52748,sid|14504|pid|10663|hfid|1684|sfid|5060|gi|667...,5060,667644755,655819,MSSHKPPEAVRRETSLQGQSMTESHCMLALSERLHDPLPGCVTASI...,0,0
52749,sid|85661|pid|63286|hfid|1696|sfid|5280|gi|667...,5280,667652271,655819,MFMLWIAAAVFGLIIIGRPLIVYLQDVKGFRKYPVQNFLSGISPLA...,0,0


In [6]:
# Create new dataframe from where there are no hits

no_hits = merged_df.loc[merged_df["sequence_y"] == 0, ["GI", "accession_x", "sequence_x", "SFID", "taxonID"]]

In [7]:
no_hits_onefamily = no_hits.drop_duplicates(subset = 'SFID', keep = 'first')
no_hits_onefamily

,GI,accession_x,sequence_x,SFID,taxonID
0,557295506,sid|90155|pid|67114|hfid|29|sfid|2|gi|55729550...,MDAGTTGILILLLLIFVLCYLVLEINRKRAQLPAGPAPWPVLGNLW...,2,38654
4026,493592559,sid|94011|pid|70441|hfid|882|sfid|107|gi|49359...,MSTVPQVVFDAFHPAHRSNPYPRYAAVREYTALYPLRPDILMATRY...,107,102897
7385,40539041,sid|96702|pid|72726|hfid|560|sfid|71|gb|AAR872...,MDDYFFLQSLLLCVAAVALLQLAKVAATMRRRPRTPPGPWRLPVIG...,71,39947
9437,558854086,sid|13873|pid|10144|hfid|153|sfid|4|gb|AHA9308...,FNQDGTDLEKIPKWGEHYPFAFPMQFSPDSTLLFVHHPAYVKPILA...,4,7998
11453,594127381,sid|86920|pid|64362|hfid|798|sfid|105|gb|EXU62...,MVDAPHPPVSMPTERSRPFNPPDGYTPLRERERITRMSYHGGATGW...,105,1158056
...,...,...,...,...,...
52743,546327208,sid|96465|pid|72513|hfid|1664|sfid|807|gi|5463...,MPTPEHVKSIFSGVLESSSAKNVARSLAVGALVGIGLRGMQSIATM...,807,2769
52744,546303216,sid|99038|pid|74742|hfid|1668|sfid|809|gi|5463...,MHVAFVFAPRLLNHSVGAQQHRRRPVCAASPPPQTEAPLPTPVAIP...,809,2769
52747,"164519833, 118380288",sid|14148|pid|10372|hfid|1677|sfid|5009|gb|ABY...,MITTILIALTFIGIALLLFKAFIQPLYRISFYTKQGLKQKFFVPFL...,5009,5911
52748,667644755,sid|14504|pid|10663|hfid|1684|sfid|5060|gi|667...,MSSHKPPEAVRRETSLQGQSMTESHCMLALSERLHDPLPGCVTASI...,5060,655819


In [8]:
# Family counts in results and no hits and save to csv

families_hits = (df_results.value_counts(df_results['SFID'])
                 .to_csv('/home/alecia/chapter3-repo/analysis/round2_missing_biocatnet/round2_biocatnet_hits_families_0p00001.txt')
)

families_nohits = (no_hits.value_counts(no_hits['SFID'])
                   .to_csv('/home/alecia/chapter3-repo/analysis/round2_missing_biocatnet/round2_biocatnet_nohits_families_0p00001.txt')
)

In [9]:
# Open these both in dataframe and merge based on SFID - drop any SFID that show up in hits

families_hits_output = pd.read_csv('/home/alecia/chapter3-repo/analysis/round2_missing_biocatnet/round2_biocatnet_hits_families_0p00001.txt')
families_nohits = pd.read_csv('/home/alecia/chapter3-repo/analysis/round2_missing_biocatnet/round2_biocatnet_nohits_families_0p00001.txt')

In [10]:
print(families_nohits)

     SFID  count
0     107   1684
1       2   1354
2     157   1122
3     125    991
4      71    925
..    ...    ...
248   259      1
249   260      1
250   750      1
251   809      1
252   807      1

[253 rows x 2 columns]


In [11]:
merged_families = (pd.merge(families_hits_output, families_nohits, how='outer', on=['SFID'])
                   .fillna(0)
                   )

In [12]:
merged_families

,SFID,count_x,count_y
0,1,389.0,366.0
1,2,2672.0,1354.0
2,3,990.0,171.0
3,4,1834.0,182.0
4,5,211.0,5.0
...,...,...,...
312,5202,0.0,2.0
313,5262,0.0,17.0
314,5280,0.0,1.0
315,5282,0.0,5.0


In [13]:
# Families where count_x = 0. That is, in the families of the sequences from the no hits list, there are zero sequences that 'hit' in that family

no_families = merged_families.loc[merged_families['count_x'] == 0, ['SFID']]

no_families['SFID'] = no_families["SFID"].astype(str)

In [14]:
nohits_seq = no_hits_onefamily.merge(no_families['SFID'], on='SFID')
nohits_seq

,GI,accession_x,sequence_x,SFID,taxonID
0,517995323,sid|61525|pid|43580|hfid|1074|sfid|152|gi|5179...,MAKKLPKDTGLDNTLKIINEAYTYVPKRLEKFGTKAFETRALGMKP...,152,53344
1,667297845,sid|107548|pid|82451|hfid|317|sfid|19|gi|66729...,MVLELLSPVSYNITSMVPEVMPVTIMPVLLLTGLLLLIWNYEGTSL...,19,482537
2,472365622,sid|20909|pid|15787|hfid|255|sfid|7|gi|4723656...,MRFAYERYVRRRPGEPPLIKGWLPYLGEALKLQKDPLGFMKTLQKQ...,7,9708
3,308229545,sid|19517|pid|14685|hfid|587|sfid|73|gb|ADO241...,MELLLLEKTLLSLFFAITLAIVISKLRGRKFKLPPGPLPVPIFGNW...,73,4682
4,465981183,sid|73287|pid|52977|hfid|260|sfid|8|gb|EMP3575...,MNRSINSKEKEHGPDSSFFPCVKMQSKTTDVSGHTGIKPVWGQNRT...,8,8469
...,...,...,...,...,...
68,546327208,sid|96465|pid|72513|hfid|1664|sfid|807|gi|5463...,MPTPEHVKSIFSGVLESSSAKNVARSLAVGALVGIGLRGMQSIATM...,807,2769
69,546303216,sid|99038|pid|74742|hfid|1668|sfid|809|gi|5463...,MHVAFVFAPRLLNHSVGAQQHRRRPVCAASPPPQTEAPLPTPVAIP...,809,2769
70,"164519833, 118380288",sid|14148|pid|10372|hfid|1677|sfid|5009|gb|ABY...,MITTILIALTFIGIALLLFKAFIQPLYRISFYTKQGLKQKFFVPFL...,5009,5911
71,667644755,sid|14504|pid|10663|hfid|1684|sfid|5060|gi|667...,MSSHKPPEAVRRETSLQGQSMTESHCMLALSERLHDPLPGCVTASI...,5060,655819


In [15]:
#Created fastafile based off of the first sequence in each family
nohits_seq_fasta = '/home/alecia/chapter3-repo/analysis/round1_missing_biocatnet/full_sequence_each_family_no_hit_r1.fasta'

with open((nohits_seq_fasta), 'w') as f:
    for index, row in nohits_seq.iterrows():
        f.write(f">{row['accession_x']}\n")
        f.write(f"{row['sequence_x']}\n")
    f.close

In [ ]:
# 176 sequences from families that did not hit at all were loaded into Jalview and ClustalO was performed.
# The following were removed:

# 15
# 19
# 35
# 39
# 57
# 74
# 152
# 241
# 301
# 307
# 347
# 393
# 413
# 505
# 524
# 645
# 726

# The I Helix of the remaining sequences was extracted and realigned in ClustalO